# Qwen2.5-7B-Instruct 全参 SFT 复现流程（云端 GPU）

本 Notebook 按顺序完成：**环境检查 → 构造 SFT 数据 → 下载基座模型 → 安装 LLaMA-Factory → 全参微调 → 推理生成提交 CSV**。

### 前置准备（手动，只需一次）
1. 新建云端 GPU 实例：7B 全参 SFT 约需 50GB 显存，推荐 **A100 80GB**；显存不足请改用 `config/llm_qwen14b_lora.yaml`（LoRA，24GB 显存可跑）或 4-bit QLoRA（16GB 可跑）。**V100/T4 不支持 bf16，需把 yaml 中的 `bf16: true` 改为 `fp16: true`**。
2. 将整个 `courseproject` 目录上传到云端 `/mnt/workspace/courseproject`（需包含 `src/`、`config/`、`data/raw/`）。
3. 从上到下依次运行本 Notebook 的全部 cell 即可复现。

In [ ]:
# Step 0. 运行环境检查：确认能看到 GPU 与 Python 版本
!nvidia-smi
!python -V

## Step 1. 检查项目文件是否齐全

确认配置文件、原始训练数据、数据构造脚本都已随项目目录上传到位（缺文件时后续步骤会直接报错）。

In [5]:
import os
path = "/mnt/workspace/courseproject"
print("config:", os.path.exists(f"{path}/config/llm_qwen7b_full.yaml"))
print("train data:", os.path.exists(f"{path}/data/raw/TRAIN/Train_reviews.csv"))
print("build script:", os.path.exists(f"{path}/src/llm/build_sft_dataset.py"))


config: True
train data: True
build script: True


## Step 2. 构造 SFT 训练数据

运行 `src/llm/build_sft_dataset.py`：把 3229 条评论的标签聚合成 Alpaca 格式（instruction/input/output），生成 `data/llm/sft_train.json` 与 LLaMA-Factory 注册文件 `dataset_info.json`。必须在项目根目录下以 `python -m` 方式运行（包内相对导入依赖）。

In [6]:
%cd /mnt/workspace/courseproject
!python -m src.llm.build_sft_dataset

/mnt/workspace/courseproject
[OK] 训练样本数: 3229
[OK] 评论数: 3229, 四元组总数: 6633
[OK] 空属性词(整体)四元组数: 4735 (71.4%)
[OK] 数据文件: /mnt/workspace/courseproject/data/llm/sft_train.json
[OK] 注册文件: /mnt/workspace/courseproject/data/llm/dataset_info.json


## Step 3. 下载 Qwen2.5-7B-Instruct 基座模型

通过 ModelScope 缓存到 `/mnt/workspace/models`（共约 15GB，首次约几分钟）。下面的「下载→列出文件→再次下载确认」三个 cell 是原始实验过程的留存：**复现时每个 cell 依次运行一遍即可**，第三个 cell 在缓存已存在时会秒结束，作用是确认下载 100% 完整。最终路径 `.../qwen--Qwen2.5-7B-Instruct/snapshots/master` 必须与 yaml 中的 `model_name_or_path` 一致（Step 5 会自动校验）。

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    'qwen/Qwen2.5-7B-Instruct',
    cache_dir='/mnt/workspace/models'
)
print("模型下载到:", model_dir)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-18 19:52:28,240 | INFO    | modelscope_hub.download | Downloading 15 files from qwen/Qwen2.5-7B-Instruct@master
Downloading:  40%|████      | 6/15 [00:13<00:19,  2.20s/file]


In [7]:
import os
d = "/mnt/workspace/models/models/qwen--Qwen2.5-7B-Instruct/snapshots/master"
for f in sorted(os.listdir(d)):
    size = os.path.getsize(os.path.join(d, f)) / 1e6
    print(f"{f:40s} {size:10.1f} MB")

.gitattributes                                  0.0 MB
LICENSE                                         0.0 MB
README.md                                       0.0 MB
config.json                                     0.0 MB
configuration.json                              0.0 MB
generation_config.json                          0.0 MB
merges.txt                                      1.7 MB
model-00001-of-00004.safetensors             3945.4 MB
model-00002-of-00004.safetensors             3864.7 MB
model-00003-of-00004.safetensors             3864.7 MB
model-00004-of-00004.safetensors             3556.4 MB
model.safetensors.index.json                    0.0 MB
tokenizer.json                                  7.0 MB
tokenizer_config.json                           0.0 MB
vocab.json                                      2.8 MB


In [8]:
from modelscope import snapshot_download
snapshot_download('qwen/Qwen2.5-7B-Instruct', cache_dir='/mnt/workspace/models')

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-18 20:02:13,694 | INFO    | modelscope_hub.download | Downloading 15 files from qwen/Qwen2.5-7B-Instruct@master
Downloading: 100%|██████████| 15/15 [00:21<00:00,  1.41s/file]


'/mnt/workspace/models/models/qwen--Qwen2.5-7B-Instruct/snapshots/master'

In [2]:
import transformers, tokenizers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.15.1
tokenizers: 0.22.2


## Step 4. 安装 LLaMA-Factory

前面直接运行 `llamafactory-cli` 报 `command not found`，是因为云端镜像未预装 LLaMA-Factory，在本 cell 安装。

> 版本兼容提示：ModelScope 镜像可能预装了互相冲突的 vllm / transformers 版本，**训练阶段这些冲突一般可忽略**。若训练时报 `transformers` 与 `tokenizers` 不兼容，可取消最后一行注释，固定到经验证的版本组合（transformers 4.55.0 + tokenizers 0.21.4）。

In [ ]:
# 方式一（推荐）：pip 直接安装 LLaMA-Factory 及其 torch/metrics 依赖组
%pip install -q "llamafactory[torch,metrics]"

# 方式二（可选）：从源码安装最新版
# !git clone https://github.com/hiyouga/LLaMA-Factory.git /mnt/workspace/LLaMA-Factory
# %cd /mnt/workspace/LLaMA-Factory
# %pip install -e ".[torch,metrics]"

!llamafactory-cli version

# 仅当训练出现 transformers/tokenizers 兼容报错时再取消注释执行：
# %pip install -q "transformers==4.55.0" "tokenizers==0.21.4"

## Step 5. 训练前配置检查

确认 yaml 中的**基座模型本地路径**（ModelScope 下载后的 `snapshots/master` 全路径）、**SFT 数据目录**、epoch 数、输出目录均正确，避免训练到一半才发现路径错误。

In [ ]:
%cd /mnt/workspace/courseproject
import os, yaml

with open('config/llm_qwen7b_full.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

model_path = cfg['model_name_or_path']
data_dir = cfg['dataset_dir']
print('model_name_or_path =', model_path)
print('  基座模型存在:', os.path.exists(model_path))
print('dataset_dir =', data_dir, '| sft_train.json 存在:', os.path.exists(os.path.join(data_dir, 'sft_train.json')))
print('num_train_epochs =', cfg['num_train_epochs'], '| output_dir =', cfg['output_dir'])
print('bf16 =', cfg.get('bf16'), '| fp16 =', cfg.get('fp16', False), '| cutoff_len =', cfg.get('cutoff_len'))
assert os.path.exists(model_path), '基座模型路径不存在，请检查 model_name_or_path'
assert os.path.exists(os.path.join(data_dir, 'sft_train.json')), '缺少 SFT 数据，请先运行 Step 2'

## Step 6. 全参 SFT 训练

- 超参：2 epoch、学习率 6e-6、余弦退火、等效 batch 32（per_device 4 × 累积 8）、gradient checkpointing、bf16；A100 80G 上约 40 分钟。
- 权重输出到 `saves/qwen7b_full_sft_2ep/`。
- **经验**：小数据（3229 条）上 3 epoch 容易过拟合、出现过度抽取；2 epoch 泛化更好，测试 F1 = 0.8042。
- yaml 中 `dataset_dir`、`output_dir` 均为相对路径，必须先 `%cd` 到项目根目录再启动训练（下面 cell 已处理）。

In [ ]:
%cd /mnt/workspace/courseproject
!llamafactory-cli train config/llm_qwen7b_full.yaml

## Step 7. 检查训练产物

全参 SFT 会把最终模型与 tokenizer 直接保存在 `output_dir`（无需合并 adapter）。

In [ ]:
%cd /mnt/workspace/courseproject
!ls -lh saves/qwen7b_full_sft_2ep
import os
_ok = os.path.exists('saves/qwen7b_full_sft_2ep/config.json')
print('训练产物就绪 ✓' if _ok else '未找到 config.json，请向上翻看训练日志排查')
assert _ok

## Step 8. 加载微调模型对测试集推理

使用项目内的 `src/llm/predict_llm.py`：自动套用 Qwen 聊天模板（不套模板会导致大量零抽取）、解析生成结果、白名单过滤并按官方格式写出 CSV。

注意：
- 默认 **transformers 后端**，云端镜像的 vllm/transformers 版本冲突时也能跑，只是速度较慢；
- 用 `--output` 写到 **`Result 7b-full-2epocns.csv`**，不要覆盖同目录下 14B-LoRA 的 `Result.csv`（融合时两个文件都要用）。

In [ ]:
%cd /mnt/workspace/courseproject
!python -m src.llm.predict_llm --model_path ./saves/qwen7b_full_sft_2ep --backend transformers --max_new_tokens 512 --output "data/llm/Result 7b-full-2epocns.csv"

In [ ]:
# 可选（快 5-10 倍）：vLLM 批量推理。
# vLLM 与 transformers 有配套版本要求（如 vLLM 0.28 需要 transformers>=5.17.0），
# 若安装后出现版本冲突，直接使用上面 transformers 后端的结果即可，无需折腾。
# !python -m src.llm.predict_llm --model_path ./saves/qwen7b_full_sft_2ep --backend vllm --max_new_tokens 512 --output "data/llm/Result 7b-full-2epocns.csv"

## Step 9. 校验结果并下载

校验 2237 个测试 id 全部覆盖、字段合法；然后通过云端 Notebook 的文件浏览器下载 `data/llm/Result 7b-full-2epocns.csv`，放回本地项目同名目录，即可运行 `python src/ensemble.py` 参与加权投票融合（该模型权重 0.8042，为最强单模型）。

In [ ]:
%cd /mnt/workspace/courseproject
import pandas as pd

out_path = 'data/llm/Result 7b-full-2epocns.csv'
df = pd.read_csv(out_path, header=None,
                 names=['id', 'AspectTerms', 'OpinionTerms', 'Categories', 'Polarities'],
                 dtype=str)
test = pd.read_csv('data/raw/TEST/Test_reviews.csv')

print('结果行数:', len(df), '| 覆盖测试 id 数:', df['id'].nunique())
assert set(df['id'].astype(int)) == set(test['id'].astype(int)), 'id 覆盖不完整！'
assert set(df['Categories']) <= set(['整体','使用体验','功效','价格','物流','气味','包装','真伪','服务','其他','成分','尺寸','新鲜度'])
assert set(df['Polarities']) <= set(['正面', '负面', '中性'])
print('id 覆盖与类别/极性白名单校验通过 ✓')
print('平均四元组数/评论: %.2f' % (len(df) / len(test)))
display(df.head(10))
print('\n类别分布:'); print(df['Categories'].value_counts())
print('\n极性分布:'); print(df['Polarities'].value_counts())